<table>
    <tr>
        <td style="text-align:left">
            <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR9ItLTT_F-3Q30cu7ZCCoKmuFGBt22pe7pNA" alt="Logo Universidad" width="300"/>
        </td>
        <td>
            Departamento de Ciencias de la Computación y de la Decisión<br>
            Facultad de Minas<br>
            Universidad Nacional de Colombia<br>
            Optimizacion e IA 2026-2S<br><br>
            Docente: Maria Constanza Torres Madronero<br>
            <br>
            Contribuciones a la guia por: <br>
            - Deimer Miranda Montoya (2023)<br>
            - Luis Fernando Becerra Monsalve (2024)
        </td>    
    </tr>
</table>

# **Optimización de hiperparametros**




In [1]:
#Importacion de librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics


In [2]:
#Carga de datos con los precios de vivienda de California
housing = fetch_california_housing()
#Organizacion de los conjuntos de entrenamiento, validacion y prueba
x_train, x_test, y_train, y_test = train_test_split(housing.data, housing.target, test_size=0.30)
x_train, x_valid, y_train, y_valid = train_test_split(x_train, y_train, test_size=0.10)

In [3]:
#Normalizacion de los datos
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_valid = scaler.transform(x_valid)
x_test = scaler.transform(x_test)

# Selección de hiperparametros

Para la selección de los hiperparametros de una red neuronal (MLP o DL) existen varias librerías. En esta práctica usares Keras Tuner. Dado que esta área aun esta en desarrollo, estas librerías se encuentran en constante desarrollo y actualización.   





In [ ]:
#!pip install keras-tuner --upgrade

In [4]:
#Importar la libreria
import keras_tuner as kt

In [7]:
# Definición del modelo

def build_model(hp):

    # Usamos el modelo Sequential
    model = keras.Sequential()
    neuronas_iniciales = x_train.shape[1:]

    # Vamos a elegir entre 8, 16 o 32 neuronas
    model.add(keras.layers.Dense(
        units=hp.Choice('units', values=[8, 16, 32, 64]),
        activation=hp.Choice('activation', values=['relu', 'tanh']),
        input_shape=neuronas_iniciales
        
        ))

    # Agregamos la capa de salida
    model.add(keras.layers.Dense(1, activation='relu'))

    # Vamos a comparar los valores 0.01, 0.001, 0.0001
    np_leatning_rate = hp.Choice(
        'learning_rate', 
        values=[1e-2, 1e-3, 1e-4]
        )

    # Incluimos la compilación del modelo
    model.compile(
        optimizer=keras.optimizers.SGD(np_leatning_rate),
        loss='mean_squared_error',
    )

    return model

In [8]:
#Ahora vamos a aplica busqueda aleatoria, buscando minimazr la perdida
#de validacion
tuner = kt.RandomSearch(
    #######
    build_model, #Funcion que define el modelo
    objective='val_loss', #Objetivo a minimizar
    max_trials=5, #Numero de pruebas a realizar
    directory='Random_Search', #Carpeta para almenzar los experimentos
    project_name='regresion'
)
#El numero de pruebas por defecto es 10, vamos a limitarlo a 5 dado
#el tiempo de computo

In [12]:
#Keras-tuner tiene diferentes funcionalidades, por ejemplo guardar cada
#modelo, o el mejor modelo en cada prueba. Una funcionalidad interesante
#es parar una prueba si el desempeno no esta mejorando, para esto usamos
#stop_early = keras.callbacks.EarlyStopping(monitor='mse',patience=3)
#ahora, realizamos la busqueda, por el tiempo de computo vamos a limitar
#el numero de epocas a 15
########

tuner.search(x_train, y_train, epochs=15, validation_data=(x_valid, y_valid))
best_model = tuner.get_best_models()[0]

c:\Users\50649\.conda\envs\ia_opt\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Una vez el modelo es entrenado, podemos pasar a la evaluación del mismo. Si no está satisfecho con los resultados, puede explorar la modificación de algunos hiperparametros, tales como el número de capas ocultas, el numero de neuronas, el paso de aprendizaje, entre otros.

In [13]:
#Evaluacion del modelo
y_pred=best_model.predict(x_test)
print("MAE: ", metrics.mean_absolute_error(y_test,y_pred))
print("MSE: ", metrics.mean_squared_error(y_test,y_pred))
print("R2:  ", metrics.r2_score(y_test,y_pred))

194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
MAE:  0.4917311790665453
MSE:  0.46046288198336677
R2:   0.6585924700215802


In [15]:
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 161 (644.00 B)

 Trainable params: 161 (644.00 B)

 Non-trainable params: 0 (0.00 B)

Dependiendo la complejidad del modelo, su entrenamiento puede tomar mas tiempo. Por lo cual, es conveniente almacenar el modelo entrenado. Para ello pueden hacer uso de la instruccion:
>>model.save("my_keras_model.h5")
Para volver a cargar el modelo, pueden usar la instruccion:
>>model = keras.models.load_model("my_keras_model.h5")

In [ ]:
#Ahora vamos a aplica optimización bayesiana
tuner = kt.BayesianOptimization(
    ####
    build_model, #Funcion que define el modelo
    objective='val_loss', #Objetivo a minimizar
    max_trials=10, #Numero de pruebas a realizar
    num_initial_points=3, # experimentos iniciales que se deben correr antes de que comience la optimización bayesiana
    directory='bayesian_tuner', #carpeta donde se almacenan los experimentos
    project_name='regresion' 
)

c:\Users\50649\.conda\envs\ia_opt\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
tuner.search(x_train, y_train, epochs=10, validation_data=(x_valid, y_valid))
best_model2 = tuner.get_best_models()[0]

Trial 10 Complete [00h 00m 13s]
val_loss: 0.382518470287323

Best val_loss So Far: 0.382518470287323
Total elapsed time: 00h 02m 27s


In [18]:
#Evaluacion del modelo
y_pred=best_model2.predict(x_test)
print("MAE: ", metrics.mean_absolute_error(y_test,y_pred))
print("MSE: ", metrics.mean_squared_error(y_test,y_pred))
print("R2:  ", metrics.r2_score(y_test,y_pred))

194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
MAE:  0.43788463416301776
MSE:  0.3801951967816606
R2:   0.7181064791068766


In [21]:
# extraer los hiperparámetros óptimos del mejor modelo
best_hps = tuner.get_best_hyperparameters()[0]
print(f"Mejor número de neuronas: {best_hps.get('units')},", 
      f"Mejor función de activación: {best_hps.get('activation')},",
      f"Mejor tasa de aprendizaje: {best_hps.get('learning_rate')},"
      )

Mejor número de neuronas: 64, Mejor función de activación: relu, Mejor tasa de aprendizaje: 0.01,


In [ ]:
import tensorflow as tf
import keras_tuner as kt
import numpy as np
from matplotlib import pyplot as plt

class CustomTuner(kt.Tuner):
      def run_trial(self, trial, train_ds, val_ds, *args, **kwargs):
        self._display.show_hyperparameter_table(trial)
        self._display.trial_number += 1
        hp = trial.hyperparameters
        model = self.hypermodel.build(trial.hyperparameters)
        
        optimizer = model.optimizer
        train_loss_metric = tf.keras.metrics.Mean()
        valid_loss_metric = tf.keras.metrics.Mean()

        loss_fn = tf.keras.losses.MeanSquaredError()
        train_ds = train_ds.batch(32)
        val_ds = val_ds.batch(32)

        def run_train_step(data):
          x = data[0]
          y = data[1]
          with tf.GradientTape() as tape:
            logits = model(x)
            loss = loss_fn(y, logits)

          gradients = tape.gradient(loss, model.trainable_variables)
          optimizer.apply_gradients(zip(gradients, model.trainable_variables))
          
          train_loss_metric.update_state(loss)
          return loss
        
        def run_valid_step(data):
          x = data[0]
          y = data[1]
          logits = model(x)
          loss = loss_fn(y, logits)

          valid_loss_metric.update_state(loss)
          return loss

        val_losses = []
        train_losses = []
        for epoch in range(5):
          tf.print("Epoch: {}".format(epoch))
          self.on_epoch_begin(trial, model, epoch, logs={})
          for batch, data in enumerate(train_ds):
              self.on_batch_begin(trial, model, batch, logs={})
              batch_loss = float(run_train_step(data))
              self.on_batch_end(trial, model, batch, logs={"loss": batch_loss})
              if batch == 6:
                loss = train_loss_metric.result()
                tf.print("Batches: {}, Loss: {}".format(batch + 1, loss))
                break
                
          for batch, data in enumerate(val_ds):
              self.on_batch_begin(trial, model, batch, logs={})
              batch_loss = float(run_valid_step(data))
              self.on_batch_end(trial, model, batch, logs={"val_loss": batch_loss})
              if batch == 6:
                loss = valid_loss_metric.result()
                tf.print("Batches: {}, Val Loss: {}".format(batch + 1, loss))
                break

          epoch_loss = train_loss_metric.result()
          self.on_epoch_end(trial, model, epoch, logs={"loss": epoch_loss})
          val_epoch_loss = valid_loss_metric.result()
          self.on_epoch_end(trial, model, epoch, logs={"val_loss": val_epoch_loss})
          
          train_losses.append(epoch_loss)
          val_losses.append(val_epoch_loss)

          train_loss_metric.reset_states()
          valid_loss_metric.reset_states()

      
        plt.plot(train_losses)
        plt.plot(val_losses)
        plt.title('Model Loss For Trial {}'.format(self._display.trial_number))
        plt.ylabel('loss')
        plt.xlabel('epoch')
        plt.legend(['train', 'val'], loc='upper left')
        plt.show()
        tf.print("Ending Trail {}".format(self._display.trial_number))
        return super(CustomTuner, self).run_trial(trial, train_ds, validation_data=val_ds, *args, **kwargs)